In [3]:
%%time
#!/usr/bin/env python3
"""
Step 2 — Authenticity Gate
==========================
Hard-drops candidates with physically impossible profiles.

Rule 1  overlapping_fulltime_roles
        Consecutive jobs overlap by > 2 months.

Rule 2  experience_exceeds_career_span
        Claimed years_of_experience > career span (months) + 36-month buffer.

Outputs
-------
artifacts/dropped_ids.parquet   candidate_id | drop_reason | drop_detail
artifacts/survivors.parquet     candidate_id
"""

from datetime import date
from pathlib import Path

import polars as pl

# ── Config ────────────────────────────────────────────────────────────────────

TODAY        = date.today()                  # fixed once for the entire run
DATASET_PATH = r"E:\Coding\Projects\redrob-ranker\dataset\candidates.jsonl"
ARTIFACTS    = Path(r"E:\Coding\Projects\redrob-ranker\dataset\artifacts")

OVERLAP_THRESHOLD_MONTHS = 2.0               # > 2 months  →  impossible
SPAN_BUFFER_MONTHS       = 36                # claimed must be ≤ span + 36 mo


# ── Helpers ───────────────────────────────────────────────────────────────────

def load_data() -> tuple[pl.DataFrame, pl.DataFrame]:
    jobs = pl.read_parquet(ARTIFACTS / "candidate_jobs.parquet")
    base = pl.read_parquet(ARTIFACTS / "candidate_base.parquet")
    return jobs, base


def prepare_jobs(jobs: pl.DataFrame) -> pl.DataFrame:
    """
    1. Cast start_date / end_date to pl.Date if they aren't already.
    2. Fill null (or is_current=True) end_dates with TODAY.
       Result column: end_date_filled  — never null after this step.
    Rows with unparseable start_date remain; their start_date will be null
    and downstream rules skip them rather than dropping on bad data.
    """
    for col in ("start_date", "end_date"):
        if jobs.schema[col] != pl.Date:
            jobs = jobs.with_columns(
                pl.col(col)
                  .cast(pl.Utf8)
                  .str.to_date(format="%Y-%m-%d", strict=False)
                  .alias(col)
            )

    jobs = jobs.with_columns(
        pl.col("is_current")
          .cast(pl.Boolean)
          .fill_null(False)
          .alias("is_current")
    )

    return jobs.with_columns(
        pl.when(pl.col("end_date").is_null() | pl.col("is_current"))
          .then(pl.lit(TODAY))
          .otherwise(pl.col("end_date"))
          .alias("end_date_filled")
    )


# ── Rule 1: overlapping full-time jobs ────────────────────────────────────────

def rule1_overlap(jobs: pl.DataFrame) -> pl.DataFrame:
    """
    For each candidate, sort jobs by start_date ASC.
    Bring the previous job's end_date_filled into each row via shift(1).over().
    Flag if (prev_end_date - this_start_date) > 2 months.

    Returns DataFrame: candidate_id | drop_reason | drop_detail
    """
    flagged = (
        jobs
        # Only rows with a solid start_date (end_date_filled is always non-null
        # after prepare_jobs, so we only guard start_date here)
        .filter(pl.col("start_date").is_not_null())

        # Sort so shift(1) gives the temporally-previous job within each candidate
        .sort(["candidate_id", "start_date"])

        # Pull previous job's end_date into this row (within each candidate)
        .with_columns(
            pl.col("end_date_filled")
              .shift(1)
              .over("candidate_id")
              .alias("prev_end_date")
        )

        # First job per candidate has no previous job — skip
        .filter(pl.col("prev_end_date").is_not_null())

        # Overlap = how far previous job's end spills into this job's start
        # Positive  →  overlap; negative  →  gap
        .with_columns(
            (
                (pl.col("prev_end_date") - pl.col("start_date"))
                .dt.total_days()
                / 30.44
            ).alias("overlap_months")
        )

        # Only genuinely impossible overlaps
        .filter(pl.col("overlap_months") > OVERLAP_THRESHOLD_MONTHS)

        # One row per candidate (worst overlap wins)
        .group_by("candidate_id")
        .agg(pl.col("overlap_months").max().alias("max_overlap_months"))

        .with_columns([
            pl.lit("overlapping_fulltime_roles").alias("drop_reason"),
            pl.concat_str([
                pl.lit("max overlap "),
                pl.col("max_overlap_months").round(1).cast(pl.Utf8),
                pl.lit(" months"),
            ]).alias("drop_detail"),
        ])
        .select(["candidate_id", "drop_reason", "drop_detail"])
    )
    return flagged


# ── Rule 2: claimed experience vs career span ─────────────────────────────────

def rule2_career_span(jobs: pl.DataFrame, base: pl.DataFrame) -> pl.DataFrame:
    """
    career_span = max(end_date_filled) - min(start_date) across all jobs.
    Flag if years_of_experience * 12 > career_span_months + 36.

    Skipped when:
      • candidate has no valid-dated jobs (nothing to measure against)
      • years_of_experience is null

    Returns DataFrame: candidate_id | drop_reason | drop_detail
    """
    span = (
        jobs
        .filter(pl.col("start_date").is_not_null())
        .group_by("candidate_id")
        .agg([
            pl.col("start_date").min().alias("earliest_start"),
            pl.col("end_date_filled").max().alias("latest_end"),
        ])
        .with_columns(
            (
                (pl.col("latest_end") - pl.col("earliest_start"))
                .dt.total_days()
                / 30.44
            ).alias("career_span_months")
        )
    )

    flagged = (
        base
        .select(["candidate_id", "years_of_experience"])
        .filter(pl.col("years_of_experience").is_not_null())
        .join(span, on="candidate_id", how="left")

        # Skip candidates with no valid job dates
        .filter(pl.col("career_span_months").is_not_null())

        .with_columns(
            (pl.col("years_of_experience") * 12).alias("claimed_months")
        )

        # Core filter: impossible overshoot
        .filter(
            pl.col("claimed_months")
            > (pl.col("career_span_months") + SPAN_BUFFER_MONTHS)
        )

        .with_columns([
            pl.lit("experience_exceeds_career_span").alias("drop_reason"),
            pl.concat_str([
                pl.lit("claimed "),
                pl.col("claimed_months").round(0).cast(pl.Int32).cast(pl.Utf8),
                pl.lit(" mo, span "),
                pl.col("career_span_months").round(0).cast(pl.Int32).cast(pl.Utf8),
                pl.lit(" mo"),
            ]).alias("drop_detail"),
        ])
        .select(["candidate_id", "drop_reason", "drop_detail"])
    )
    return flagged


# ── Main ──────────────────────────────────────────────────────────────────────

def main() -> None:
    print(f"TODAY = {TODAY}  (fixed for this run)")
    print(f"Loading parquets from {ARTIFACTS} …\n")

    jobs_raw, base = load_data()
    jobs = prepare_jobs(jobs_raw)

    total_candidates = base["candidate_id"].n_unique()
    print(f"  candidate_jobs rows      : {len(jobs):>10,}")
    print(f"  total candidates in base : {total_candidates:>10,}\n")

    # ── Rule 1 ────────────────────────────────────────────────────────────────
    print("Running Rule 1 — overlapping full-time jobs …")
    dropped_r1 = rule1_overlap(jobs)
    print(f"  → {len(dropped_r1):,} candidates flagged\n")

    # ── Rule 2 ────────────────────────────────────────────────────────────────
    print("Running Rule 2 — experience vs career span …")
    dropped_r2 = rule2_career_span(jobs, base)
    print(f"  → {len(dropped_r2):,} candidates flagged\n")

    # ── Merge (R1 reason takes priority if both fire) ─────────────────────────
    dropped_all = (
        pl.concat([dropped_r1, dropped_r2])
        .unique(subset=["candidate_id"], keep="first")
    )
    both_rules = (
        set(dropped_r1["candidate_id"].to_list())
        & set(dropped_r2["candidate_id"].to_list())
    )

    # ── Survivors ─────────────────────────────────────────────────────────────
    survivors = (
        base
        .select("candidate_id")
        .filter(~pl.col("candidate_id").is_in(dropped_all["candidate_id"]))
    )

    # ── Write artifacts ───────────────────────────────────────────────────────
    ARTIFACTS.mkdir(parents=True, exist_ok=True)
    dropped_all.write_parquet(ARTIFACTS / "dropped_ids.parquet")
    survivors.write_parquet(ARTIFACTS / "survivors.parquet")

    # ── Summary ───────────────────────────────────────────────────────────────
    print("=" * 50)
    print(f"  Total candidates          {total_candidates:>8,}")
    print(f"  Dropped — Rule 1 overlap  {len(dropped_r1):>8,}")
    print(f"  Dropped — Rule 2 span     {len(dropped_r2):>8,}")
    print(f"  Dropped — both rules      {len(both_rules):>8,}")
    print(f"  Dropped — unique total    {len(dropped_all):>8,}")
    print(f"  Survivors                 {len(survivors):>8,}")
    print("=" * 50)
    print(f"\ndropped_ids.parquet  →  {ARTIFACTS / 'dropped_ids.parquet'}")
    print(f"survivors.parquet    →  {ARTIFACTS / 'survivors.parquet'}")


if __name__ == "__main__":
    main()

TODAY = 2026-06-25  (fixed for this run)
Loading parquets from E:\Coding\Projects\redrob-ranker\dataset\artifacts …

  candidate_jobs rows      :    300,171
  total candidates in base :    100,000

Running Rule 1 — overlapping full-time jobs …
  → 0 candidates flagged

Running Rule 2 — experience vs career span …
  → 25 candidates flagged

  Total candidates           100,000
  Dropped — Rule 1 overlap         0
  Dropped — Rule 2 span           25
  Dropped — both rules             0
  Dropped — unique total          25
  Survivors                   99,975

dropped_ids.parquet  →  E:\Coding\Projects\redrob-ranker\dataset\artifacts\dropped_ids.parquet
survivors.parquet    →  E:\Coding\Projects\redrob-ranker\dataset\artifacts\survivors.parquet
CPU times: total: 219 ms
Wall time: 232 ms


<timed exec>:235: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.


In [6]:
import polars as pl
from pathlib import Path

ARTIFACTS = Path(r"E:\Coding\Projects\redrob-ranker\dataset\artifacts")

base = pl.read_parquet(ARTIFACTS / "candidate_base.parquet")
jobs = pl.read_parquet(ARTIFACTS / "candidate_jobs.parquet")
dropped_all = pl.read_parquet(ARTIFACTS / "dropped_ids.parquet")
survivors = pl.read_parquet(ARTIFACTS / "survivors.parquet")

In [7]:
dropped_preview = dropped_all.join(base, on="candidate_id", how="left")
survivor_preview = survivors.join(base, on="candidate_id", how="left")

print(dropped_preview.head(10))
print(survivor_preview.head(10))

shape: (10, 13)
┌──────────────┬──────────────┬──────────────┬─────────────┬───┬─────────────┬─────────────┬─────────────┬─────────────┐
│ candidate_id ┆ drop_reason  ┆ drop_detail  ┆ anonymized_ ┆ … ┆ current_tit ┆ current_com ┆ current_com ┆ current_ind │
│ ---          ┆ ---          ┆ ---          ┆ name        ┆   ┆ le          ┆ pany        ┆ pany_size   ┆ ustry       │
│ str          ┆ str          ┆ str          ┆ ---         ┆   ┆ ---         ┆ ---         ┆ ---         ┆ ---         │
│              ┆              ┆              ┆ str         ┆   ┆ str         ┆ str         ┆ str         ┆ str         │
╞══════════════╪══════════════╪══════════════╪═════════════╪═══╪═════════════╪═════════════╪═════════════╪═════════════╡
│ CAND_0033131 ┆ experience_e ┆ claimed 152  ┆ Vikram      ┆ … ┆ Operations  ┆ Pied Piper  ┆ 11-50       ┆ Software    │
│              ┆ xceeds_caree ┆ mo, span 33  ┆ Chatterjee  ┆   ┆ Manager     ┆             ┆             ┆             │
│              ┆

In [8]:
dropped_all.head(10)

candidate_id,drop_reason,drop_detail
str,str,str
"""CAND_0033131""","""experience_exceeds_career_span""","""claimed 152 mo, span 33 mo"""
"""CAND_0093331""","""experience_exceeds_career_span""","""claimed 193 mo, span 87 mo"""
"""CAND_0024752""","""experience_exceeds_career_span""","""claimed 179 mo, span 48 mo"""
"""CAND_0052478""","""experience_exceeds_career_span""","""claimed 149 mo, span 17 mo"""
"""CAND_0071115""","""experience_exceeds_career_span""","""claimed 198 mo, span 69 mo"""
"""CAND_0055992""","""experience_exceeds_career_span""","""claimed 203 mo, span 80 mo"""
"""CAND_0013536""","""experience_exceeds_career_span""","""claimed 169 mo, span 56 mo"""
"""CAND_0003430""","""experience_exceeds_career_span""","""claimed 164 mo, span 16 mo"""
"""CAND_0086808""","""experience_exceeds_career_span""","""claimed 137 mo, span 20 mo"""


In [9]:
survivors.head(10)

candidate_id
str
"""CAND_0000001"""
"""CAND_0000002"""
"""CAND_0000003"""
"""CAND_0000004"""
"""CAND_0000005"""
"""CAND_0000006"""
"""CAND_0000007"""
"""CAND_0000008"""
"""CAND_0000009"""


In [10]:
len(base), len(survivors), len(dropped_all)

(100000, 99975, 25)

In [11]:
len(survivors) + len(dropped_all)

100000

In [15]:
import polars as pl
from pathlib import Path
from datetime import date

# ── Paths / config ───────────────────────────────────────────────────────────

TODAY = date.today()
ARTIFACTS = Path(r"E:\Coding\Projects\redrob-ranker\dataset\artifacts")

DAYS_PER_MONTH = 30.44
OVERLAP_FLOOR_MONTHS = 2.0
SPAN_BUFFER_MAX_MONTHS = 36
SPAN_BUFFER_RATIO = 0.5


# ── Load outputs from Step 1 ─────────────────────────────────────────────────

base = pl.read_parquet(ARTIFACTS / "candidate_base.parquet")
jobs = pl.read_parquet(ARTIFACTS / "candidate_jobs.parquet")


# ── Minimal preparation ──────────────────────────────────────────────────────

for col in ("start_date", "end_date"):
    if jobs.schema[col] != pl.Date:
        jobs = jobs.with_columns(
            pl.col(col)
            .cast(pl.Utf8)
            .str.to_date(format="%Y-%m-%d", strict=False)
            .alias(col)
        )

if "is_current" in jobs.columns:
    dtype = jobs["is_current"].dtype

    if dtype == pl.Boolean:
        jobs = jobs.with_columns(
            pl.col("is_current").fill_null(False)
        )
    elif dtype == pl.Utf8:
        jobs = jobs.with_columns(
            pl.col("is_current")
            .str.to_lowercase()
            .str.strip_chars()
            .is_in(["true", "1", "yes"])
            .alias("is_current")
        )
    else:
        jobs = jobs.with_columns(
            pl.col("is_current")
            .cast(pl.Boolean, strict=False)
            .fill_null(False)
            .alias("is_current")
        )
else:
    jobs = jobs.with_columns(pl.lit(False).alias("is_current"))

jobs = jobs.with_columns(
    pl.when(pl.col("end_date").is_null() | pl.col("is_current"))
    .then(pl.lit(TODAY))
    .otherwise(pl.col("end_date"))
    .alias("end_date_filled")
)


# ── Rule 1: overlapping full-time jobs ───────────────────────────────────────

if "employment_type" in jobs.columns:
    jobs_ft = jobs.filter(
        (
            pl.col("employment_type")
            .str.to_lowercase()
            .str.strip_chars()
            .str.replace_all("-", "_")
            .str.replace_all(" ", "_")
        ) == "full_time"
    )
else:
    jobs_ft = jobs

dropped_r1 = (
    jobs_ft
    .filter(pl.col("start_date").is_not_null())
    .sort(["candidate_id", "start_date", "end_date_filled"])
    .with_columns(
        pl.col("end_date_filled")
        .shift(1)
        .over("candidate_id")
        .alias("prev_end_date")
    )
    .filter(pl.col("prev_end_date").is_not_null())
    .with_columns(
        (
            (pl.col("prev_end_date") - pl.col("start_date"))
            .dt.total_days()
            / DAYS_PER_MONTH
        ).alias("overlap_months")
    )
    .filter(pl.col("overlap_months") > OVERLAP_FLOOR_MONTHS)
    .group_by("candidate_id")
    .agg(pl.col("overlap_months").max().alias("max_overlap_months"))
    .with_columns([
        pl.lit("overlapping_fulltime_roles").alias("drop_reason"),
        pl.concat_str([
            pl.lit("max overlap "),
            pl.col("max_overlap_months").round(1).cast(pl.Utf8),
            pl.lit(" months"),
        ]).alias("drop_detail"),
    ])
    .select(["candidate_id", "drop_reason", "drop_detail"])
)


# ── Rule 2: claimed experience exceeds career span ───────────────────────────

span = (
    jobs
    .filter(pl.col("start_date").is_not_null())
    .group_by("candidate_id")
    .agg([
        pl.col("start_date").min().alias("earliest_start"),
        pl.col("end_date_filled").max().alias("latest_end"),
    ])
    .with_columns(
        (
            (pl.col("latest_end") - pl.col("earliest_start"))
            .dt.total_days()
            / DAYS_PER_MONTH
        ).alias("career_span_months")
    )
)

dropped_r2 = (
    base
    .select(["candidate_id", "years_of_experience"])
    .filter(pl.col("years_of_experience").is_not_null())
    .join(span, on="candidate_id", how="left")
    .filter(pl.col("career_span_months").is_not_null())
    .with_columns(
        (pl.col("years_of_experience") * 12).alias("claimed_months")
    )
    .with_columns(
        pl.min_horizontal(
            pl.lit(float(SPAN_BUFFER_MAX_MONTHS)),
            pl.col("career_span_months") * SPAN_BUFFER_RATIO,
        ).alias("effective_buffer")
    )
    .filter(
        pl.col("claimed_months")
        > pl.col("career_span_months") + pl.col("effective_buffer")
    )
    .with_columns([
        pl.lit("experience_exceeds_career_span").alias("drop_reason"),
        pl.concat_str([
            pl.lit("claimed "),
            pl.col("claimed_months").round(0).cast(pl.Int32).cast(pl.Utf8),
            pl.lit(" mo, span "),
            pl.col("career_span_months").round(0).cast(pl.Int32).cast(pl.Utf8),
            pl.lit(" mo, buffer "),
            pl.col("effective_buffer").round(0).cast(pl.Int32).cast(pl.Utf8),
            pl.lit(" mo"),
        ]).alias("drop_detail"),
    ])
    .select(["candidate_id", "drop_reason", "drop_detail"])
)


# ── Export audit samples ─────────────────────────────────────────────────────

def export_audit_sample(dropped_df: pl.DataFrame, rule_name: str, n: int = 5) -> None:
    if len(dropped_df) == 0:
        print(f"\nNo candidates dropped for {rule_name}. Skipping export.")
        return

    sample_n = min(n, len(dropped_df))

    sample_candidates = (
        dropped_df
        .join(base, on="candidate_id", how="left")
        .sample(n=sample_n, seed=42)
    )

    sample_jobs = (
        jobs
        .join(sample_candidates.select("candidate_id"), on="candidate_id", how="inner")
        .sort(["candidate_id", "start_date", "end_date_filled"])
    )

    candidate_csv = ARTIFACTS / f"audit_{rule_name}_5_candidates.csv"
    candidate_json = ARTIFACTS / f"audit_{rule_name}_5_candidates.json"
    jobs_csv = ARTIFACTS / f"audit_{rule_name}_5_jobs.csv"
    jobs_json = ARTIFACTS / f"audit_{rule_name}_5_jobs.json"

    sample_candidates.write_csv(candidate_csv)
    sample_candidates.write_json(candidate_json)

    sample_jobs.write_csv(jobs_csv)
    sample_jobs.write_json(jobs_json)

    print(f"\nExported audit sample for {rule_name}:")
    print(f"  Candidates CSV  → {candidate_csv}")
    print(f"  Candidates JSON → {candidate_json}")
    print(f"  Jobs CSV        → {jobs_csv}")
    print(f"  Jobs JSON       → {jobs_json}")

    print("\nCandidate sample:")
    print(sample_candidates)

    print("\nJob history sample:")
    print(sample_jobs)


export_audit_sample(dropped_r1, "overlap", n=5)
export_audit_sample(dropped_r2, "extra_experience", n=5)


# ── Summary ──────────────────────────────────────────────────────────────────

print("\n" + "=" * 52)
print("Audit generation summary")
print("=" * 52)
print(f"Original candidates       : {base['candidate_id'].n_unique():>8,}")
print(f"Original job rows         : {len(jobs):>8,}")
print(f"Rule 1 overlap drops      : {len(dropped_r1):>8,}")
print(f"Rule 2 extra exp drops    : {len(dropped_r2):>8,}")


No candidates dropped for overlap. Skipping export.

Exported audit sample for extra_experience:
  Candidates CSV  → E:\Coding\Projects\redrob-ranker\dataset\artifacts\audit_extra_experience_5_candidates.csv
  Candidates JSON → E:\Coding\Projects\redrob-ranker\dataset\artifacts\audit_extra_experience_5_candidates.json
  Jobs CSV        → E:\Coding\Projects\redrob-ranker\dataset\artifacts\audit_extra_experience_5_jobs.csv
  Jobs JSON       → E:\Coding\Projects\redrob-ranker\dataset\artifacts\audit_extra_experience_5_jobs.json

Candidate sample:
shape: (5, 13)
┌──────────────┬──────────────┬──────────────┬─────────────┬───┬─────────────┬─────────────┬─────────────┬─────────────┐
│ candidate_id ┆ drop_reason  ┆ drop_detail  ┆ anonymized_ ┆ … ┆ current_tit ┆ current_com ┆ current_com ┆ current_ind │
│ ---          ┆ ---          ┆ ---          ┆ name        ┆   ┆ le          ┆ pany        ┆ pany_size   ┆ ustry       │
│ str          ┆ str          ┆ str          ┆ ---         ┆   ┆ ---   

In [16]:
import polars as pl
from pathlib import Path
from datetime import date

# ── Config ──────────────────────────────────────────────────────────────────

TODAY = date.today()
ARTIFACTS = Path(r"E:\Coding\Projects\redrob-ranker\dataset\artifacts")

DAYS_PER_MONTH = 30.44
SPAN_BUFFER_MAX_MONTHS = 36
SPAN_BUFFER_RATIO = 0.5

N = 10  # change to 5 if you only want top 5


# ── Load ────────────────────────────────────────────────────────────────────

base = pl.read_parquet(ARTIFACTS / "candidate_base.parquet")
jobs = pl.read_parquet(ARTIFACTS / "candidate_jobs.parquet")


# ── Prepare jobs minimally ──────────────────────────────────────────────────

for col in ("start_date", "end_date"):
    if jobs.schema[col] != pl.Date:
        jobs = jobs.with_columns(
            pl.col(col)
            .cast(pl.Utf8)
            .str.to_date(format="%Y-%m-%d", strict=False)
            .alias(col)
        )

if jobs["is_current"].dtype == pl.Boolean:
    jobs = jobs.with_columns(pl.col("is_current").fill_null(False))
elif jobs["is_current"].dtype == pl.Utf8:
    jobs = jobs.with_columns(
        pl.col("is_current")
        .str.to_lowercase()
        .str.strip_chars()
        .is_in(["true", "1", "yes"])
        .alias("is_current")
    )
else:
    jobs = jobs.with_columns(
        pl.col("is_current")
        .cast(pl.Boolean, strict=False)
        .fill_null(False)
        .alias("is_current")
    )

jobs = jobs.with_columns(
    pl.when(pl.col("end_date").is_null() | pl.col("is_current"))
    .then(pl.lit(TODAY))
    .otherwise(pl.col("end_date"))
    .alias("end_date_filled")
)


# ── Recompute Rule 2 with excess_months ──────────────────────────────────────

span = (
    jobs
    .filter(pl.col("start_date").is_not_null())
    .group_by("candidate_id")
    .agg([
        pl.col("start_date").min().alias("earliest_start"),
        pl.col("end_date_filled").max().alias("latest_end"),
    ])
    .with_columns(
        (
            (pl.col("latest_end") - pl.col("earliest_start"))
            .dt.total_days()
            / DAYS_PER_MONTH
        ).alias("career_span_months")
    )
)

worst_extra_exp = (
    base
    .select(["candidate_id", "years_of_experience"])
    .filter(pl.col("years_of_experience").is_not_null())
    .join(span, on="candidate_id", how="left")
    .filter(pl.col("career_span_months").is_not_null())
    .with_columns([
        (pl.col("years_of_experience") * 12).alias("claimed_months"),
        pl.min_horizontal(
            pl.lit(float(SPAN_BUFFER_MAX_MONTHS)),
            pl.col("career_span_months") * SPAN_BUFFER_RATIO,
        ).alias("effective_buffer"),
    ])
    .with_columns(
        (
            pl.col("career_span_months") + pl.col("effective_buffer")
        ).alias("allowed_months")
    )
    .with_columns(
        (
            pl.col("claimed_months") - pl.col("allowed_months")
        ).alias("excess_months")
    )
    .filter(pl.col("excess_months") > 0)
    .sort("excess_months", descending=True)
    .head(N)
    .with_columns([
        pl.lit("experience_exceeds_career_span").alias("drop_reason"),
        pl.concat_str([
            pl.lit("claimed "),
            pl.col("claimed_months").round(0).cast(pl.Int32).cast(pl.Utf8),
            pl.lit(" mo, span "),
            pl.col("career_span_months").round(0).cast(pl.Int32).cast(pl.Utf8),
            pl.lit(" mo, buffer "),
            pl.col("effective_buffer").round(0).cast(pl.Int32).cast(pl.Utf8),
            pl.lit(" mo, excess "),
            pl.col("excess_months").round(0).cast(pl.Int32).cast(pl.Utf8),
            pl.lit(" mo"),
        ]).alias("drop_detail"),
    ])
)


# ── Add base profile details ─────────────────────────────────────────────────

worst_extra_exp_candidates = (
    worst_extra_exp
    .join(base, on="candidate_id", how="left")
)

worst_extra_exp_jobs = (
    jobs
    .join(
        worst_extra_exp_candidates.select("candidate_id"),
        on="candidate_id",
        how="inner",
    )
    .sort(["candidate_id", "start_date", "end_date_filled"])
)


# ── Export with clear names ──────────────────────────────────────────────────

candidate_csv = ARTIFACTS / "worst_offenders_extra_experience_candidates.csv"
candidate_json = ARTIFACTS / "worst_offenders_extra_experience_candidates.json"

jobs_csv = ARTIFACTS / "worst_offenders_extra_experience_jobs.csv"
jobs_json = ARTIFACTS / "worst_offenders_extra_experience_jobs.json"

worst_extra_exp_candidates.write_csv(candidate_csv)
worst_extra_exp_candidates.write_json(candidate_json)

worst_extra_exp_jobs.write_csv(jobs_csv)
worst_extra_exp_jobs.write_json(jobs_json)


# ── Display ─────────────────────────────────────────────────────────────────

print("\nWorst extra-experience offenders:")
print(worst_extra_exp_candidates)

print("\nJobs for worst extra-experience offenders:")
print(worst_extra_exp_jobs)

print("\nWrote:")
print(f"  {candidate_csv}")
print(f"  {candidate_json}")
print(f"  {jobs_csv}")
print(f"  {jobs_json}")


Worst extra-experience offenders:
shape: (10, 21)
┌──────────────┬──────────────┬──────────────┬────────────┬───┬──────────────┬─────────────┬─────────────┬─────────────┐
│ candidate_id ┆ years_of_exp ┆ earliest_sta ┆ latest_end ┆ … ┆ current_titl ┆ current_com ┆ current_com ┆ current_ind │
│ ---          ┆ erience      ┆ rt           ┆ ---        ┆   ┆ e            ┆ pany        ┆ pany_size   ┆ ustry       │
│ str          ┆ ---          ┆ ---          ┆ date       ┆   ┆ ---          ┆ ---         ┆ ---         ┆ ---         │
│              ┆ f64          ┆ date         ┆            ┆   ┆ str          ┆ str         ┆ str         ┆ str         │
╞══════════════╪══════════════╪══════════════╪════════════╪═══╪══════════════╪═════════════╪═════════════╪═════════════╡
│ CAND_0003430 ┆ 13.7         ┆ 2025-03-03   ┆ 2026-06-25 ┆ … ┆ Business     ┆ Infosys     ┆ 10001+      ┆ IT Services │
│              ┆              ┆              ┆            ┆   ┆ Analyst      ┆             ┆          

In [17]:
base.head()

candidate_id,anonymized_name,headline,summary,location,country,years_of_experience,current_title,current_company,current_company_size,current_industry
str,str,str,str,str,str,f64,str,str,str,str
"""CAND_0000001""","""Ira Vora""","""Backend Engineer | SQL, Spark,…","""Software / data professional w…","""Toronto""","""Canada""",6.9,"""Backend Engineer""","""Mindtree""","""10001+""","""IT Services"""
"""CAND_0000002""","""Saanvi Sethi""","""Operations Manager | 12.5+ yrs…","""Professional with 12.5+ years …","""Chennai, Tamil Nadu""","""India""",12.5,"""Operations Manager""","""Wipro""","""10001+""","""IT Services"""
"""CAND_0000003""","""Yash Agarwal""","""Customer Support | 1.1+ yrs ex…","""Professional with 1.1+ years o…","""Austin""","""USA""",1.1,"""Customer Support""","""TCS""","""10001+""","""IT Services"""
"""CAND_0000004""","""Anil Bose""","""Marketing Manager | Driving bu…","""Professional with 3.8+ years o…","""Sydney""","""Australia""",3.8,"""Marketing Manager""","""Dunder Mifflin""","""201-500""","""Paper Products"""
"""CAND_0000005""","""Aisha Sethi""","""Accountant | Helping teams sca…","""Professional with 11.0+ years …","""Gurgaon, Haryana""","""India""",11.0,"""Accountant""","""Stark Industries""","""1001-5000""","""Manufacturing"""


In [18]:
jobs.head()

candidate_id,job_index,company,title,start_date,end_date,duration_months,is_current,industry,company_size,description,end_date_filled
str,i64,str,str,date,date,i64,bool,str,str,str,date
"""CAND_0000001""",0,"""Mindtree""","""Backend Engineer""",2024-03-08,null,27,true,"""IT Services""","""10001+""","""Implemented streaming data pip…",2026-06-25
"""CAND_0000001""",1,"""Dunder Mifflin""","""Analytics Engineer""",2019-07-03,2024-01-08,55,false,"""Paper Products""","""201-500""","""Built and maintained data pipe…",2024-01-08
"""CAND_0000002""",0,"""Wipro""","""Operations Manager""",2022-11-14,null,43,true,"""IT Services""","""10001+""","""Customer support team lead at …",2026-06-25
"""CAND_0000002""",1,"""Wipro""","""Operations Manager""",2021-09-13,2022-11-07,14,false,"""IT Services""","""10001+""","""Mechanical engineering design …",2022-11-07
"""CAND_0000002""",2,"""Acme Corp""","""Marketing Manager""",2017-03-08,2021-08-14,54,false,"""Manufacturing""","""201-500""","""Content writing and SEO strate…",2021-08-14


In [19]:
jobs.columns

['candidate_id',
 'job_index',
 'company',
 'title',
 'start_date',
 'end_date',
 'duration_months',
 'is_current',
 'industry',
 'company_size',
 'description',
 'end_date_filled']